# Exploring expert cross-reference memberships

This notebook explores how experts cross-reference source vegetation units (`VEGCODE`, e.g. TASVEG 5.0) to IUCN Global Ecosystem Typology functional groups (`efg`). By default it reads the first-round combined table produced by `read-input-xref-data.ipynb` (experts AK, AL, JB, MGT), but any long-format table with the same four columns can be swapped in at section 2.

| Section | What it does |
|---|---|
| 1. Set up | Load packages and the shared plot style |
| 2. Get data | **One variable** (`DATA_INPUT`) chooses the data; it is then read, validated and checked |
| 3. Matrix table | Reshape to a complete unit × expert × target membership matrix and compute pairwise distances |
| 4. Overall exploration | Complete similarity and dissimilarity heatmaps across all experts and units |
| 5. Other exploration | Expert distributions, target usage, contested units, single-unit and single-expert views, interactive browser |
| 6. Notes | The process step by step, and a summary computed from the current data |

**Kernel:** R (IRkernel). Run top to bottom.

## 1. Set up

Load packages (installing any that are missing) and define the shared theme and colours.

In [ ]:
packages_needed <- c("here", "readr", "readxl", "dplyr", "tidyr", "ggplot2", "tibble", "plotly")
packages_missing <- packages_needed[!vapply(packages_needed, requireNamespace, logical(1), quietly = TRUE)]
if (length(packages_missing) > 0) {
  install.packages(packages_missing, repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(invisible(lapply(packages_needed, library, character.only = TRUE)))

here::i_am("viz_explore_ars.ipynb")
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 110)

# Shared chart styling.
theme_wp3 <- function() {
  ggplot2::theme_minimal(base_size = 12) +
    ggplot2::theme(
      plot.title = ggplot2::element_text(color = "#17313B", face = "bold", size = 15),
      plot.subtitle = ggplot2::element_text(color = "#52666F", margin = ggplot2::margin(b = 10)),
      axis.title = ggplot2::element_text(color = "#334B55", face = "bold"),
      axis.text = ggplot2::element_text(color = "#52666F"),
      panel.grid.minor = ggplot2::element_blank(),
      panel.grid.major = ggplot2::element_line(color = "#E6ECEF", linewidth = 0.35),
      strip.background = ggplot2::element_rect(fill = "#EDF2F4", color = NA),
      strip.text = ggplot2::element_text(color = "#17313B", face = "bold"),
      legend.position = "bottom",
      legend.title = ggplot2::element_text(color = "#334B55", face = "bold"),
      plot.background = ggplot2::element_rect(fill = "white", color = NA)
    )
}

# A restrained, color-blind-aware palette for a small number of experts.
expert_palette <- c("#355C7D", "#B36A4C", "#5B8063", "#84659A", "#A7833F", "#477C82")
teal_low <- "#F1F7F6"; teal_high <- "#176B6E"
rust_low <- "#FFF4E9"; rust_high <- "#C75B45"
# White labels on dark tiles, slate labels on light tiles.
label_colour_scale <- ggplot2::scale_colour_manual(values = c(`TRUE` = "white", `FALSE` = "#17313B"), guide = "none")

R.version.string

## 2. Get data

`DATA_INPUT` is the **only** thing to change to explore different data. It can be:

- a path to a `.csv` file (default: the first-round combined output);
- a path to an `.xlsx`/`.xls` file (first sheet is read);
- a data frame already in the R session, e.g. `xref_table_experts` from `read-input-xref-data.ipynb`, or a new round you've just built.

Whatever you supply needs the columns `VEGCODE`, `efg`, `memb` and `expert` (one row per expert/unit/target). Membership can be on a 0–1 or 0–100 scale.

In [2]:
DATA_INPUT <- here::here("output-XREFs", "First-round-output.csv")
# DATA_INPUT <- here::here("output-XREFs", "Second-round-output.csv")   # another file
# DATA_INPUT <- xref_table_experts                                       # a data frame in memory

### 2.1 Read

`read_membership_input()` picks the right reader for whatever `DATA_INPUT` is.

In [ ]:
read_membership_input <- function(input) {
  if (is.data.frame(input)) return(tibble::as_tibble(input))
  if (!is.character(input) || length(input) != 1) stop("DATA_INPUT must be a file path or a data frame.")
  if (!file.exists(input)) stop("File not found: ", input)
  ext <- tolower(tools::file_ext(input))
  switch(ext,
         csv = readr::read_csv(input, show_col_types = FALSE),
         xlsx = , xls = readxl::read_excel(input),
         stop("Unsupported file type '.", ext, "'; use .csv, .xlsx, or pass a data frame."))
}

raw <- read_membership_input(DATA_INPUT)
data_label <- if (is.data.frame(DATA_INPUT)) "data frame supplied in session" else basename(DATA_INPUT)
message("Read ", nrow(raw), " rows from ", data_label)
dplyr::glimpse(raw)

### 2.2 Harmonise and validate

Structural problems (missing columns, blank codes, duplicate records, an unusable scale) stop the notebook. Content issues are **reported, not fixed**, so they can be taken back to the experts:

- rows with a missing membership value (dropped from the analysis);
- target codes that don't match the usual `XX0.0` pattern (possible typos);
- units where an expert's memberships don't sum to 1.

In [ ]:
required_fields <- c("VEGCODE", "efg", "memb", "expert")
missing_fields <- setdiff(required_fields, names(raw))
if (length(missing_fields) > 0) stop("Missing required fields: ", paste(missing_fields, collapse = ", "))

xref_all <- raw |>
  dplyr::select(dplyr::all_of(required_fields)) |>
  dplyr::mutate(
    VEGCODE = trimws(as.character(VEGCODE)),
    efg = trimws(as.character(efg)),
    expert = trimws(as.character(expert)),
    memb = suppressWarnings(as.numeric(memb))
  )

if (any(is.na(xref_all$VEGCODE) | xref_all$VEGCODE == "")) stop("VEGCODE has missing/blank values.")
if (any(is.na(xref_all$efg) | xref_all$efg == "")) stop("efg has missing/blank values.")
if (any(is.na(xref_all$expert) | xref_all$expert == "")) stop("expert has missing/blank values.")
if (any(xref_all$memb < 0, na.rm = TRUE)) stop("memb must be non-negative.")
if (max(xref_all$memb, na.rm = TRUE) > 1 && max(xref_all$memb, na.rm = TRUE) <= 100) xref_all$memb <- xref_all$memb / 100
if (max(xref_all$memb, na.rm = TRUE) > 1) stop("memb must use a consistent 0–1 or 0–100 scale.")
if (anyDuplicated(xref_all[c("expert", "VEGCODE", "efg")])) stop("Duplicate rows found for expert/VEGCODE/efg.")

check_missing_memb <- xref_all |> dplyr::filter(is.na(memb))
check_efg_codes <- xref_all |> dplyr::filter(!grepl("^[A-Z]+[0-9]+\\.[0-9]+$", efg))
check_totals <- xref_all |>
  dplyr::group_by(VEGCODE, expert) |>
  dplyr::summarise(total_membership = sum(memb), n_targets = dplyr::n(), .groups = "drop") |>
  dplyr::filter(is.na(total_membership) | abs(total_membership - 1) > 0.001)

xref <- xref_all |> dplyr::filter(!is.na(memb))
experts <- sort(unique(xref$expert))
units <- sort(unique(xref$VEGCODE))
if (length(experts) < 2) stop("At least two experts are needed to compare memberships.")
expert_colors <- stats::setNames(rep(expert_palette, length.out = length(experts)), experts)

message(nrow(xref), " records | ", length(units), " units | ", dplyr::n_distinct(xref$efg),
        " target groups | experts: ", paste(experts, collapse = ", "))
message(nrow(check_missing_memb), " row(s) with missing membership dropped; ",
        nrow(check_efg_codes), " non-standard target code(s); ",
        nrow(check_totals), " unit/expert total(s) not equal to 1.")

In [ ]:
print(check_missing_memb)
print(check_efg_codes)
print(check_totals)

## 3. Matrix table

The long table is reshaped into a **complete membership matrix**: one row per unit and expert, one column per target group. For each unit, the matrix covers every target that *any* expert assigned to it. A target an expert didn't list is set to **0** (not assigned); units an expert never assessed are left out rather than filled.

Two derived tables are built from it and used in sections 4 and 5:

- `pairwise`: for every unit and every pair of experts, the **dissimilarity** between their profiles — half the sum of absolute differences, i.e. the share of membership that would have to move to make them identical (0 = identical, 1 = no overlap). **Similarity** = `1 − dissimilarity`.
- `unit_disagreement`: per unit, the mean, minimum and maximum dissimilarity across expert pairs.

These are descriptive comparisons, not formal inter-rater reliability statistics. Profiles that don't sum to 1 can push dissimilarity slightly past 1; colour scales are clipped to 0–1.

In [ ]:
# Long, complete profiles: one row per unit × expert × target, absent targets = 0.
profiles <- xref |>
  dplyr::group_by(VEGCODE) |>
  tidyr::complete(expert, efg, fill = list(memb = 0)) |>
  dplyr::ungroup()

# Wide matrix table: rows = unit × expert, columns = target groups.
efg_order <- sort(unique(xref$efg))
membership_matrix <- profiles |>
  tidyr::pivot_wider(names_from = efg, values_from = memb) |>
  dplyr::select(VEGCODE, expert, dplyr::any_of(efg_order)) |>
  dplyr::arrange(VEGCODE, expert)

message(nrow(membership_matrix), " rows (unit × expert) × ", length(efg_order), " target columns. ",
        "NA = target not used by any expert for that unit; 0 = used by another expert but not this one.")
head(membership_matrix, 12)

In [ ]:
# Pairwise expert distances for each unit.
pairwise <- profiles |>
  dplyr::inner_join(profiles, by = c("VEGCODE", "efg"), suffix = c("_a", "_b"),
                    relationship = "many-to-many") |>
  dplyr::filter(expert_a < expert_b) |>
  dplyr::group_by(VEGCODE, expert_a, expert_b) |>
  dplyr::summarise(dissimilarity = 0.5 * sum(abs(memb_a - memb_b)), .groups = "drop") |>
  dplyr::mutate(similarity = 1 - dissimilarity, pair = paste(expert_a, expert_b, sep = " – "))

unit_disagreement <- pairwise |>
  dplyr::group_by(VEGCODE) |>
  dplyr::summarise(
    mean_dissimilarity = mean(dissimilarity),
    min_dissimilarity = min(dissimilarity),
    max_dissimilarity = max(dissimilarity),
    n_pairs = dplyr::n(),
    .groups = "drop"
  ) |>
  dplyr::arrange(dplyr::desc(mean_dissimilarity), VEGCODE)

message(sum(unit_disagreement$max_dissimilarity < 0.001), " of ", nrow(unit_disagreement),
        " units have identical profiles from every expert.")
head(unit_disagreement, 10)

## 4. Overall exploration: similarity and dissimilarity

### 4.1 Expert × expert heatmaps

Complete expert-by-expert matrices, averaged over every unit both experts assessed. The diagonal is each expert compared with themselves (similarity 1, dissimilarity 0). Labels give the mean and the share of units where the two profiles are identical.

In [ ]:
pair_summary <- pairwise |>
  dplyr::group_by(expert_a, expert_b) |>
  dplyr::summarise(
    n_units = dplyr::n(),
    mean_similarity = mean(similarity),
    mean_dissimilarity = mean(dissimilarity),
    median_similarity = stats::median(similarity),
    share_identical = mean(dissimilarity < 0.001),
    .groups = "drop"
  )

# Mirror the pairs and add the diagonal so the matrix is complete.
pair_matrix <- dplyr::bind_rows(
  pair_summary |> dplyr::rename(expert_x = expert_a, expert_y = expert_b),
  pair_summary |> dplyr::rename(expert_x = expert_b, expert_y = expert_a),
  tibble::tibble(expert_x = experts, expert_y = experts, mean_similarity = 1, mean_dissimilarity = 0,
                 share_identical = 1)
) |>
  dplyr::mutate(
    expert_x = factor(expert_x, levels = experts), expert_y = factor(expert_y, levels = experts),
    sim_label = sprintf("%.2f\n(%.0f%% identical)", mean_similarity, 100 * share_identical),
    dis_label = sprintf("%.2f", mean_dissimilarity)
  )

expert_matrix_plot <- function(fill_col, label_col, low, high, title, subtitle, fill_name) {
  ggplot2::ggplot(pair_matrix, ggplot2::aes(x = expert_x, y = expert_y, fill = .data[[fill_col]])) +
    ggplot2::geom_tile(color = "white", linewidth = 0.8) +
    ggplot2::geom_text(ggplot2::aes(label = .data[[label_col]], colour = .data[[fill_col]] > 0.6), size = 3.3) +
    label_colour_scale +
    ggplot2::scale_fill_gradient(low = low, high = high, limits = c(0, 1), oob = scales::squish) +
    ggplot2::coord_equal() +
    ggplot2::labs(title = title, subtitle = subtitle, x = NULL, y = NULL, fill = fill_name) +
    theme_wp3() + ggplot2::theme(panel.grid.major = ggplot2::element_blank())
}

options(repr.plot.width = 7, repr.plot.height = 6.5)
expert_matrix_plot("mean_similarity", "sim_label", teal_low, teal_high,
                   "Expert similarity across all units", "1 = identical profiles", "Mean similarity")
expert_matrix_plot("mean_dissimilarity", "dis_label", rust_low, rust_high,
                   "Expert dissimilarity across all units", "0 = identical, 1 = no shared membership", "Mean dissimilarity")

pair_summary

### 4.2 Unit × expert-pair dissimilarity heatmap

Every unit (rows) against every pair of experts (columns), so all the comparisons behind 4.1 are visible at once. Rows are ordered from most to least contested; the right-hand column is the unit's mean. White = the pair agree exactly.

In [ ]:
unit_order <- rev(unit_disagreement$VEGCODE)
pair_levels <- c(unique(pairwise$pair), "Mean")
complete_dissimilarity <- dplyr::bind_rows(
  pairwise |> dplyr::select(VEGCODE, pair, dissimilarity),
  unit_disagreement |> dplyr::transmute(VEGCODE, pair = "Mean", dissimilarity = mean_dissimilarity)
) |>
  dplyr::mutate(VEGCODE = factor(VEGCODE, levels = unit_order), pair = factor(pair, levels = pair_levels))

options(repr.plot.width = 8, repr.plot.height = max(6, 0.14 * length(units) + 2))
ggplot2::ggplot(complete_dissimilarity, ggplot2::aes(x = pair, y = VEGCODE, fill = dissimilarity)) +
  ggplot2::geom_tile(color = "white", linewidth = 0.25) +
  ggplot2::geom_vline(xintercept = length(pair_levels) - 0.5, colour = "#17313B", linewidth = 0.6) +
  ggplot2::scale_fill_gradient(low = "white", high = rust_high, limits = c(0, 1), oob = scales::squish) +
  ggplot2::scale_x_discrete(position = "top") +
  ggplot2::labs(title = "Dissimilarity for every unit and expert pair",
                subtitle = "Rows ordered from most (top) to least contested",
                x = NULL, y = "Unit (VEGCODE)", fill = "Dissimilarity") +
  theme_wp3() +
  ggplot2::theme(panel.grid.major = ggplot2::element_blank(), axis.text.y = ggplot2::element_text(size = 6.5),
                 axis.text.x = ggplot2::element_text(size = 9, colour = "#17313B", face = "bold"),
                 plot.subtitle = ggplot2::element_text(color = "#52666F", margin = ggplot2::margin(b = 14)))

In [ ]:
# Both in one: upper triangle = similarity, lower triangle = dissimilarity.
# One colour scale on similarity; dissimilarity = 1 − similarity, so each tile
# is coloured the same way whichever number it shows.
agreement_matrix <- pair_matrix |>
  dplyr::mutate(
    ix = as.integer(expert_x), iy = as.integer(expert_y),
    part = dplyr::case_when(ix == iy ~ "diagonal", iy > ix ~ "similarity", TRUE ~ "dissimilarity"),
    fill_value = ifelse(part == "diagonal", NA, mean_similarity),
    label = dplyr::case_when(
      part == "similarity" ~ sprintf("similarity %.2f\n%.0f%% identical\nn = %d",
                                     mean_similarity, 100 * share_identical, n_units),
      part == "dissimilarity" ~ sprintf("dissimilarity %.2f\nn = %d", mean_dissimilarity, n_units),
      TRUE ~ sprintf("%s\n%d units", expert_x, n_units)
    ),
    dark_tile = !is.na(fill_value) & abs(fill_value - 0.5) > 0.3
  )

options(repr.plot.width = 7.5, repr.plot.height = 7.5)
ggplot2::ggplot(agreement_matrix, ggplot2::aes(x = expert_x, y = expert_y, fill = fill_value)) +
  ggplot2::geom_tile(colour = "white", linewidth = 0.8) +
  ggplot2::geom_text(ggplot2::aes(label = label, colour = dark_tile), size = 3.1) +
  label_colour_scale +
  ggplot2::scale_fill_gradientn(
    colours = c(rust_high, "#F7F4EF", teal_high), limits = c(0, 1), oob = scales::squish,
    na.value = "#EDF2F4", breaks = c(0, 0.5, 1),
    labels = c("similarity 0\ndissimilarity 1", "0.5\n0.5", "similarity 1\ndissimilarity 0"),
    guide = ggplot2::guide_colourbar(reverse = TRUE)   # dissimilarity end (rust) on the right
  ) +
  ggplot2::coord_equal() +
  ggplot2::labs(title = "Expert similarity and dissimilarity across shared units",
                subtitle = "Upper triangle = mean similarity · lower triangle = mean dissimilarity (1 − similarity)\nn = units both experts assessed",
                x = NULL, y = NULL, fill = NULL) +
  theme_wp3() +
  ggplot2::theme(panel.grid.major = ggplot2::element_blank(),
                 legend.key.width = ggplot2::unit(2.2, "cm"))

### 5. Interactive unit browser

Use the dropdown in the output to step through the `n_contested` most contested units (ordered as in 5.3). Each view is the expert × target membership heatmap for that unit; hover for values.

In [19]:
browser_units <- as.character(dplyr::slice_head(unit_disagreement, n = n_contested)$VEGCODE)
unit_browser <- plotly::plot_ly()
unit_axes <- list()

for (k in seq_along(browser_units)) {
  u <- browser_units[[k]]
  d <- profiles |> dplyr::filter(VEGCODE == u)
  targets <- sort(unique(d$efg))
  z <- matrix(NA_real_, nrow = length(experts), ncol = length(targets), dimnames = list(experts, targets))
  z[cbind(match(d$expert, experts), match(d$efg, targets))] <- d$memb
  hover <- matrix(paste0("Expert: ", rep(experts, times = length(targets)),
                         "<br>Target: ", rep(targets, each = length(experts)),
                         "<br>Membership: ", ifelse(is.na(z), "not assessed", sprintf("%.2f", z))),
                  nrow = length(experts))
  unit_browser <- plotly::add_trace(
    unit_browser, type = "heatmap", x = targets, y = experts, z = z,
    text = hover, hoverinfo = "text", zmin = 0, zmax = 1, xgap = 2, ygap = 2,
    colorscale = list(list(0, teal_low), list(1, teal_high)),
    colorbar = list(title = "Membership"), showscale = k == 1, visible = k == 1, name = u
  )
  unit_axes[[u]] <- targets
}

unit_title <- function(u) {
  row <- unit_disagreement[unit_disagreement$VEGCODE == u, ]
  sprintf("%s — mean dissimilarity %.2f (pair range %.2f–%.2f)",
          u, row$mean_dissimilarity, row$min_dissimilarity, row$max_dissimilarity)
}

unit_buttons <- lapply(seq_along(browser_units), function(k) {
  u <- browser_units[[k]]
  list(label = u, method = "update",
       args = list(list(visible = seq_along(browser_units) == k,
                        showscale = seq_along(browser_units) == k),
                   list(title = list(text = unit_title(u)),
                        xaxis = list(title = list(text = "Target group (efg)"), type = "category",
                                     categoryorder = "array", categoryarray = unit_axes[[u]],
                                     automargin = TRUE, showgrid = FALSE))))
})

unit_browser |>
  plotly::layout(
    title = list(text = unit_title(browser_units[[1]])),
    xaxis = list(title = "Target group (efg)", type = "category", categoryorder = "array",
                 categoryarray = unit_axes[[browser_units[[1]]]], automargin = TRUE, showgrid = FALSE),
    yaxis = list(title = "Expert", type = "category", automargin = TRUE, showgrid = FALSE),
    updatemenus = list(list(type = "dropdown", active = 0, x = 0, y = 1.2,
                            xanchor = "left", yanchor = "top", buttons = unit_buttons)),
    margin = list(l = 80, r = 30, t = 110, b = 80),
    paper_bgcolor = "white", plot_bgcolor = "white",
    font = list(family = "Arial, sans-serif", size = 12, color = "#52666F"),
    hoverlabel = list(bgcolor = "white", bordercolor = "#D4DEE2",
                      font = list(family = "Arial, sans-serif", size = 12, color = "#17313B"))
  ) |>
  plotly::config(displaylogo = FALSE, responsive = TRUE)

## 6. Notes

### Process

1. **Set up** — load packages and a shared plot style (section 1).
2. **Get data** — `DATA_INPUT` points to a CSV/Excel file or a data frame; it is read into one long table with `VEGCODE`, `efg`, `memb`, `expert` (2.1).
3. **Validate** — stop on structural errors (missing columns, blanks, duplicates, bad scale); rescale 0–100 to 0–1; report missing memberships, non-standard target codes and totals ≠ 1 without changing them. Rows with missing membership are dropped (2.2).
4. **Build the matrix** — for each unit, complete every expert's profile across all targets any expert used, filling unlisted targets with 0; pivot to a unit × expert by target matrix (3).
5. **Compare** — for every unit and expert pair, dissimilarity = ½ × Σ|difference| (0 identical, 1 no overlap), similarity = 1 − dissimilarity; summarise per unit and per expert pair (3).
6. **Explore overall** — expert × expert similarity/dissimilarity heatmaps and the full unit × pair dissimilarity heatmap (4).
7. **Explore in detail** — expert value distributions, target usage, most contested units, one unit, one expert vs the rest, and an interactive browser (5).

### Summary

The cell below writes the summary from the current data, so it stays correct when `DATA_INPUT` changes.

In [20]:
most_similar <- pair_summary |> dplyr::slice_max(mean_similarity, n = 1, with_ties = FALSE)
least_similar <- pair_summary |> dplyr::slice_min(mean_similarity, n = 1, with_ties = FALSE)
expert_mean_sim <- pairwise |>
  tidyr::pivot_longer(c(expert_a, expert_b), values_to = "expert") |>
  dplyr::group_by(expert) |>
  dplyr::summarise(mean_similarity = mean(similarity), .groups = "drop") |>
  dplyr::arrange(mean_similarity)
n_identical <- sum(unit_disagreement$max_dissimilarity < 0.001)
top_contested <- head(unit_disagreement, 5)

summary_points <- c(
  sprintf("Data: %s — %d records, %d units, %d target groups, %d experts (%s).",
          data_label, nrow(xref), length(units), length(efg_order), length(experts), paste(experts, collapse = ", ")),
  sprintf("Data checks: %d missing membership value(s), %d non-standard target code(s)%s, %d unit/expert total(s) not equal to 1.",
          nrow(check_missing_memb), nrow(check_efg_codes),
          if (nrow(check_efg_codes) > 0) paste0(" (", paste(unique(check_efg_codes$efg), collapse = ", "), ")") else "",
          nrow(check_totals)),
  sprintf("Targets per unit: %s.",
          paste(sprintf("%s %.1f", expert_summary$expert, expert_summary$mean_targets_per_unit), collapse = ", ")),
  sprintf("Overall agreement: mean similarity across all expert pairs and units is %.2f; %d of %d units (%.0f%%) are identical for every expert.",
          mean(pairwise$similarity), n_identical, length(units), 100 * n_identical / length(units)),
  sprintf("Most similar pair: %s & %s (mean similarity %.2f, identical on %.0f%% of units).",
          most_similar$expert_a, most_similar$expert_b, most_similar$mean_similarity, 100 * most_similar$share_identical),
  sprintf("Least similar pair: %s & %s (mean similarity %.2f, identical on %.0f%% of units).",
          least_similar$expert_a, least_similar$expert_b, least_similar$mean_similarity, 100 * least_similar$share_identical),
  sprintf("Expert who differs most from the others on average: %s (mean similarity %.2f).",
          expert_mean_sim$expert[[1]], expert_mean_sim$mean_similarity[[1]]),
  sprintf("Most contested units: %s.",
          paste(sprintf("%s (%.2f)", top_contested$VEGCODE, top_contested$mean_dissimilarity), collapse = ", "))
)
cat(paste0("• ", summary_points), sep = "\n")

### Caveats

- Similarity and dissimilarity describe differences between membership profiles. They don't measure ecological similarity or formal inter-rater reliability.
- Don't weight experts or average them into a final membership table without a defensible project method.

### Glossary

| Term | Column | Definition |
|---|---|---|
| **Cross-reference (xref)** | — | A table that links units of one classification to units of another, recording how strongly each pair corresponds. |
| **Source** | — | The classification being cross-referenced *from*. Here it is TASVEG 5.0, the Tasmanian vegetation mapping. There is one source in this data, so the notebook doesn't need a `source` column. |
| **Source unit / unit** | `VEGCODE` | One mapping unit of the source classification, identified by its TASVEG code (e.g. `AHL`). Each unit is assessed separately. |
| **Target** | `efg` | The classification being cross-referenced *to*: an Ecosystem Functional Group (EFG) of the IUCN Global Ecosystem Typology. Codes are realm + biome + group, e.g. `T2.6` (terrestrial), `F2.2` (freshwater), `MT1.3` (marine–terrestrial shoreline), `TF1.6` (terrestrial–freshwater wetland), `MFT1.3` (marine–freshwater–terrestrial). |
| **Expert** | `expert` | The person who made the assessment, identified by initials (AK, AL, JB, MGT). Each expert works independently, and their assessments are compared, not merged. |
| **Membership** | `memb` | An expert's estimate of how much of a source unit corresponds to a target group, from 0 (none) to 1 (all of it). One unit can be split across several targets, so an expert's memberships for a unit should sum to 1. |
| **Assigned / not assigned** | — | A target is *assigned* when the expert lists it for the unit with a membership above 0. A target listed by another expert but not by this one is *not assigned* and is treated as 0. |
| **Not assessed** | — | The expert gave no record for the unit at all. This is different from "not assigned" and is never filled with 0. |
| **Record** | one row | One expert's membership for one unit and one target. Long format means one record per row. |
| **Profile** | — | An expert's full set of memberships for one unit, across every target that any expert used for that unit. |
| **Matrix table** | `membership_matrix` | The data reshaped wide: one row per unit and expert, one column per target group. |
| **Dissimilarity** | `dissimilarity` | How different two experts' profiles are for a unit: ½ × the sum of absolute differences, i.e. the share of membership that would have to move to make them identical. 0 = identical, 1 = no shared membership. |
| **Similarity** | `similarity` | 1 − dissimilarity. 1 = identical profiles, 0 = no overlap. |
| **Identical** | `share_identical` | Two profiles with a dissimilarity of essentially 0 (below 0.001). |
| **Expert pair** | `expert_a`, `expert_b` | Two experts compared on the same unit. With four experts there are six pairs. |
| **Contested unit** | `unit_disagreement` | A unit where experts disagree. Units are ranked by their mean dissimilarity across all expert pairs. |
| **Data checks** | `check_*` | Problems that are reported but not fixed: missing membership values, target codes that don't match the `XX0.0` pattern, and totals per unit and expert that aren't 1. |

In [21]:
sessionInfo()